In [0]:
import urllib.request
import os

# Define source URL and destination volume path
source_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
volume_path = "/Volumes/nyc_taxi_lakehouse/bronze/raw_taxi_data"
file_path = f"{volume_path}/yellow_tripdata_2024-01.parquet"

# Ensure directory exists and download file
os.makedirs(volume_path, exist_ok=True)
print(f"Downloading NYC Taxi dataset to {file_path}...")
urllib.request.urlretrieve(source_url, file_path)
print("Download complete.")

Download complete.


In [0]:
# Configure Auto Loader stream
bronze_checkpoint = "abfss://medallion-data@boodystorage.dfs.core.windows.net/checkpoints/bronze"
raw_data_path = "/Volumes/nyc_taxi_lakehouse/bronze/raw_taxi_data"

bronze_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", f"{bronze_checkpoint}/schema")
    .load(raw_data_path)
)

# Append audit metadata using Unity Catalog supported syntax
from pyspark.sql.functions import current_timestamp, col

bronze_with_metadata = (
    bronze_stream
    .withColumn("_ingestion_time", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path"))
)

query = (
    bronze_with_metadata.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{bronze_checkpoint}/data")
    .trigger(availableNow=True)
    .toTable("nyc_taxi_lakehouse.bronze.taxi_trips_raw")
)

query.awaitTermination()
print("Bronze ingestion complete!")

Bronze ingestion complete!


In [0]:
%sql
SELECT 
    VendorID, 
    tpep_pickup_datetime, 
    tpep_dropoff_datetime, 
    passenger_count, 
    trip_distance, 
    fare_amount, 
    _ingestion_time, 
    _source_file 
FROM nyc_taxi_lakehouse.bronze.taxi_trips_raw 
LIMIT 10;

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,fare_amount,_ingestion_time,_source_file
2,2024-01-01T00:57:55.000,2024-01-01T01:17:43.000,1,1.72,17.7,2026-10-05T16:54:26.439Z,/Volumes/nyc_taxi_lakehouse/bronze/raw_taxi_data/yellow_tripdata_2024-01.parquet
1,2024-01-01T00:03:00.000,2024-01-01T00:09:36.000,1,1.8,10.0,2026-10-05T16:54:26.439Z,/Volumes/nyc_taxi_lakehouse/bronze/raw_taxi_data/yellow_tripdata_2024-01.parquet
1,2024-01-01T00:17:06.000,2024-01-01T00:35:01.000,1,4.7,23.3,2026-10-05T16:54:26.439Z,/Volumes/nyc_taxi_lakehouse/bronze/raw_taxi_data/yellow_tripdata_2024-01.parquet
1,2024-01-01T00:36:38.000,2024-01-01T00:44:56.000,1,1.4,10.0,2026-10-05T16:54:26.439Z,/Volumes/nyc_taxi_lakehouse/bronze/raw_taxi_data/yellow_tripdata_2024-01.parquet
1,2024-01-01T00:46:51.000,2024-01-01T00:52:57.000,1,0.8,7.9,2026-10-05T16:54:26.439Z,/Volumes/nyc_taxi_lakehouse/bronze/raw_taxi_data/yellow_tripdata_2024-01.parquet
1,2024-01-01T00:54:08.000,2024-01-01T01:26:31.000,1,4.7,29.6,2026-10-05T16:54:26.439Z,/Volumes/nyc_taxi_lakehouse/bronze/raw_taxi_data/yellow_tripdata_2024-01.parquet
2,2024-01-01T00:49:44.000,2024-01-01T01:15:47.000,2,10.82,45.7,2026-10-05T16:54:26.439Z,/Volumes/nyc_taxi_lakehouse/bronze/raw_taxi_data/yellow_tripdata_2024-01.parquet
1,2024-01-01T00:30:40.000,2024-01-01T00:58:40.000,0,3.0,25.4,2026-10-05T16:54:26.439Z,/Volumes/nyc_taxi_lakehouse/bronze/raw_taxi_data/yellow_tripdata_2024-01.parquet
2,2024-01-01T00:26:01.000,2024-01-01T00:54:12.000,1,5.44,31.0,2026-10-05T16:54:26.439Z,/Volumes/nyc_taxi_lakehouse/bronze/raw_taxi_data/yellow_tripdata_2024-01.parquet
2,2024-01-01T00:28:08.000,2024-01-01T00:29:16.000,1,0.04,3.0,2026-10-05T16:54:26.439Z,/Volumes/nyc_taxi_lakehouse/bronze/raw_taxi_data/yellow_tripdata_2024-01.parquet
